# Brand Color Palette Extraction

Extract dominant brand colours and their visual share from a design asset.

**Portfolio category:** Computer vision

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Generate an owned demonstration image

In [ ]:
height, width = 120, 240
image = np.ones((height, width, 3), dtype=float)
colours = np.array([
    [0.03, 0.22, 0.45],
    [0.00, 0.55, 0.75],
    [0.98, 0.45, 0.12],
    [0.95, 0.80, 0.18],
    [0.12, 0.68, 0.42],
    [0.92, 0.94, 0.97],
])
boundaries = [0, 55, 100, 145, 185, 215, 240]
for colour, left, right in zip(colours, boundaries[:-1], boundaries[1:]):
    image[:, left:right] = colour
image = np.clip(image + rng.normal(0, 0.018, image.shape), 0, 1)
n_colours = 6

plt.imshow(image)
plt.axis('off')
plt.title('Original demonstration image')
plt.tight_layout()

## 3. Pixel quality checks

In [ ]:
pixels = image.reshape(-1, 3)
display(pd.DataFrame(pixels, columns=["red", "green", "blue"]).describe().T)
print("Image shape:", image.shape)
print("Pixel range:", float(pixels.min()), "to", float(pixels.max()))

## 4. Learn the colour palette

In [ ]:
sample_size = min(12000, len(pixels))
sample = pixels[rng.choice(len(pixels), size=sample_size, replace=False)]
model = MiniBatchKMeans(
    n_clusters=n_colours,
    n_init=20,
    batch_size=2048,
    random_state=RANDOM_STATE,
).fit(sample)
labels = model.predict(pixels)
palette_values = np.clip(model.cluster_centers_, 0, 1)
reconstructed = palette_values[labels].reshape(image.shape)

## 5. Quantisation quality

In [ ]:
mse = mean_squared_error(pixels, reconstructed.reshape(-1, 3))
psnr = 20 * np.log10(1.0 / np.sqrt(max(mse, 1e-12)))
proportions = pd.Series(labels).value_counts(normalize=True).sort_index()
display(pd.Series({
    "palette_colours": n_colours,
    "mse": mse,
    "psnr_db": psnr,
    "largest_colour_share": proportions.max(),
}).to_frame("value"))

## 6. Compare original and quantised images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[1].imshow(reconstructed)
axes[1].set_title(f"Quantised to {n_colours} colours")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

## 7. Palette with proportions

In [ ]:
order = proportions.sort_values(ascending=False).index
palette_table = pd.DataFrame(palette_values[order], columns=["red", "green", "blue"])
palette_table["share"] = proportions.loc[order].to_numpy()
palette_table["hex"] = [
    "#{:02X}{:02X}{:02X}".format(*(np.rint(rgb * 255).astype(int)))
    for rgb in palette_values[order]
]
display(palette_table.round(3))
fig, ax = plt.subplots(figsize=(10, 2))
left = 0
for rgb, share in zip(palette_values[order], proportions.loc[order]):
    ax.barh([0], [share], left=left, color=rgb, height=0.7)
    left += share
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_title("Learned palette and visual share")
plt.tight_layout()

## 8. Storage interpretation

In [ ]:
original_bits = image.size * 8
label_bits = len(pixels) * np.ceil(np.log2(n_colours))
palette_bits = n_colours * 3 * 8
display(pd.Series({
    "original_rgb_bits": original_bits,
    "approx_quantised_bits": label_bits + palette_bits,
    "approx_compression_ratio": original_bits / (label_bits + palette_bits),
}).to_frame("value"))

## 9. Key findings

Choose palette size from the quality-storage trade-off and always evaluate on the actual asset type.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For brand color palette extraction,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.